# PRNU-v2 reference-free binary usefulness gate

Run this notebook from **Runtime > Run all** in a Colab GPU runtime. Before any expensive Task 2 preparation, it requires the label-free PREMIER device-signal gate to pass at the same native 256 px crop used by the binary runtime vector. If local Task 8B source data is already staged, it can produce that validation automatically; otherwise it reuses a verified 256 px report from Drive and stops early with an actionable message when neither is available. It then rebuilds the selected matched-clean and independent robustness views on Colab-local storage, restores the three controlled-RINE parent checkpoints from Drive, extracts the reference-free PRNU-v2 vector, trains the PRNU-only diagnostic and RINE+PRNU candidate for seeds 42/43/44, applies the locked retention gate, and syncs only durable outputs. Deliberately downscaled robustness cells remain in evaluation with explicit PRNU validity/confidence masks. It never reads `final_test`, never treats known-device PCE as an authenticity score, and never copies the 19,460-image transform cache back to Drive.

Expected Drive inputs:

- `MyDrive/hackathon_data/raw/sid_set/images`
- `MyDrive/cya-techjam26/artifacts/task2/fixed_q96_manifest.csv`
- `MyDrive/cya-techjam26/artifacts/task2/source_manifest_split.csv`
- `MyDrive/cya-techjam26/artifacts/robustness/train-controlled-rine/seed_{42,43,44}`
- either a validated 256 px report at `MyDrive/cya-techjam26/artifacts/task8b_v2/audits/prnu_v2_signal_validation.json`, or locally staged PREMIER data plus the Task 8B prepared manifest


In [14]:
import subprocess
from pathlib import Path

PROJECT = Path('/content/cya-techjam26')
REPO_URL = 'https://github.com/maxi-cmyk/cya-techjam26.git'
if PROJECT.is_dir():
    git_result = subprocess.run(
        ['git', 'pull', '--ff-only'], cwd=PROJECT, text=True
    )
else:
    git_result = subprocess.run(
        ['git', 'clone', REPO_URL, str(PROJECT)], text=True
    )
assert git_result.returncode == 0, 'Repository clone/pull failed.'
required_files = (
    PROJECT / 'scripts/extract_prnu_runtime_v2.py',
    PROJECT / 'scripts/train_prnu_runtime_v2.py',
    PROJECT / 'scripts/run_robustness_fusion.py',
    PROJECT / 'scripts/compare_robustness_candidate.py',
    PROJECT / 'src/cya_detector/features/prnu_runtime_v2.py',
)
missing_files = [str(path.relative_to(PROJECT)) for path in required_files if not path.is_file()]
assert not missing_files, (
    'The checkout does not contain the Notebook 08 implementation. Commit and push '
    f'the PRNU-v2 changes first. Missing: {missing_files}'
)
print('Repository ready:', PROJECT)


Repository ready: /content/cya-techjam26


In [15]:
from google.colab import drive

drive.mount('/content/drive')
print('Drive mounted.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


In [16]:
import csv
import json
import shutil
import subprocess
from pathlib import Path

PROJECT = Path('/content/cya-techjam26')

def run_command(*args):
    result = subprocess.run([str(value) for value in args], cwd=PROJECT, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, args))}")
    return result

run_command('make', 'install-colab')
import torch
assert torch.cuda.is_available(), (
    'Notebook 08 requires a GPU runtime. In Colab select Runtime > Change runtime '
    'type > GPU, then use Runtime > Run all.'
)
print('GPU:', torch.cuda.get_device_name(0))
run_command('make', 'smoke-bootstrap')
run_command(
    'python', '-m', 'unittest',
    'tests.test_prnu_runtime_v2',
    'tests.test_robustness_training',
    '-v',
)
print('Environment and targeted tests passed.')

# Fail before the expensive robustness-bank rebuild unless the v2 estimator
# has passed a label-free device test at this exact 256 px protocol.
DRIVE_ARTIFACTS = Path('/content/drive/MyDrive/cya-techjam26/artifacts')
LOCAL_TASK8B = PROJECT / 'artifacts/task8b'
LOCAL_TASK8B_V2 = PROJECT / 'artifacts/task8b_v2'
DRIVE_TASK8B_V2 = DRIVE_ARTIFACTS / 'task8b_v2'
signal_relative = Path('audits/prnu_v2_signal_validation.json')
signal_256_relative = Path('audits/prnu_v2_signal_validation_256.json')

def valid_256_signal_report(path):
    if not path.is_file():
        return None
    report = json.loads(path.read_text())
    if report.get('crop_size') != 256 or not report.get('signal_validated'):
        return None
    assert not report.get('binary_authenticity_labels_used', False)
    assert not report.get('selection_or_heldout_rows_read', False)
    return report

drive_signal_candidates = (
    DRIVE_TASK8B_V2 / signal_256_relative,
    DRIVE_TASK8B_V2 / signal_relative,
)
drive_signal_source = next(
    (path for path in drive_signal_candidates if valid_256_signal_report(path) is not None),
    None,
)
signal_report = valid_256_signal_report(drive_signal_source) if drive_signal_source else None
if signal_report is not None:
    local_signal_path = LOCAL_TASK8B_V2 / signal_relative
    local_signal_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_signal_source, local_signal_path)
    shutil.copy2(drive_signal_source, LOCAL_TASK8B_V2 / signal_256_relative)
    print('Reusing validated 256 px PREMIER device-signal report from Drive.')
else:
    old_signal_path = DRIVE_TASK8B_V2 / signal_relative
    if old_signal_path.is_file():
        old_signal = json.loads(old_signal_path.read_text())
        if old_signal.get('crop_size') == 512:
            old_512_path = DRIVE_TASK8B_V2 / 'audits/prnu_v2_signal_validation_512.json'
            old_512_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(old_signal_path, old_512_path)
            old_fingerprints = DRIVE_TASK8B_V2 / 'fingerprints'
            archived_fingerprints = DRIVE_TASK8B_V2 / 'fingerprints_crop_512'
            if old_fingerprints.is_dir() and not archived_fingerprints.exists():
                shutil.copytree(old_fingerprints, archived_fingerprints)
            print('Archived the earlier 512 px validation evidence before rerunning.')
    task8b_raw = Path('/content/hackathon_data/raw/task8b')
    manifest_candidates = (
        LOCAL_TASK8B / 'manifests/source_manifest_split.csv',
        DRIVE_ARTIFACTS / 'task8b/manifests/source_manifest_split.csv',
    )
    source_manifest = next((path for path in manifest_candidates if path.is_file()), None)
    if source_manifest is None or not (task8b_raw / 'premier').is_dir():
        raise RuntimeError(
            'A validated 256 px PREMIER report is required before binary training. '
            'The existing 512 px report is not reused. Stage Task 8B data with '
            'Notebook 06 through its preparation cell, then rerun Notebook 08; '
            'this check occurs before rebuilding or retraining Task 2.'
        )
    with source_manifest.open(newline='') as stream:
        reader = csv.DictReader(stream)
        fields = reader.fieldnames
        task8b_rows = list(reader)
    assert fields and 'relative_path' in fields and 'image_path' in fields
    missing_task8b = []
    for row in task8b_rows:
        row['image_path'] = str(task8b_raw / row['relative_path'])
        if not Path(row['image_path']).is_file():
            missing_task8b.append(row['image_path'])
    assert not missing_task8b, f'Task 8B source files are incomplete: {missing_task8b[:5]}'
    local_manifest = LOCAL_TASK8B / 'manifests/source_manifest_split_local.csv'
    local_manifest.parent.mkdir(parents=True, exist_ok=True)
    with local_manifest.open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(task8b_rows)
    run_command(
        'make', 'task8b-v2-prnu-validate',
        f'TASK8B_MANIFEST={local_manifest}', 'PRNU_V2_CROP_SIZE=256',
    )
    signal_report = valid_256_signal_report(LOCAL_TASK8B_V2 / signal_relative)
    assert signal_report is not None, 'The 256 px PREMIER device-signal gate did not pass.'
    shutil.copytree(LOCAL_TASK8B_V2, DRIVE_TASK8B_V2, dirs_exist_ok=True)
    print('Validated the 256 px PREMIER device signal and saved it to Drive.')
print('PRNU-v2 256 px signal AUC:', signal_report['roc_auc'])


GPU: NVIDIA A100-SXM4-40GB
Environment and targeted tests passed.
Reusing validated 256 px PREMIER device-signal report from Drive.
PRNU-v2 256 px signal AUC: 0.8592845276689085


In [17]:
import csv
import hashlib
import json
import shutil
import subprocess
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

PROJECT = Path('/content/cya-techjam26')
PRIOR = Path('/content/drive/MyDrive/cya-techjam26/artifacts')
DURABLE = PRIOR / 'robustness'
ROBUSTNESS = PROJECT / 'artifacts/robustness'
SEEDS = (42, 43, 44)
fixed_manifest = PRIOR / 'task2/fixed_q96_manifest.csv'
source_manifest = PRIOR / 'task2/source_manifest_split.csv'
drive_images = Path('/content/drive/MyDrive/hackathon_data/raw/sid_set/images')
local_images = Path('/content/hackathon_data/raw/sid_set/images')
regenerated_manifest = PROJECT / 'artifacts/task2/fixed_q96_manifest_regenerated.csv'

def run_command(*args):
    result = subprocess.run([str(value) for value in args], cwd=PROJECT, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, args))}")
    return result

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

for required in (fixed_manifest, source_manifest, drive_images, DURABLE):
    assert required.exists(), f'Required Drive input not found: {required}'

with fixed_manifest.open(newline='') as stream:
    fixed_rows = list(csv.DictReader(stream))
assert len(fixed_rows) == 2000, f'Expected 2,000 selected rows, found {len(fixed_rows)}'
assert Counter(row['label'] for row in fixed_rows) == {'authentic': 1000, 'ai_generated': 1000}
selected_source_ids = {row['source_id'] for row in fixed_rows}
assert len(selected_source_ids) == len(fixed_rows), 'Selected source IDs are not unique'

def stage_selected_source(row):
    filename = Path(row['source_path']).name
    source = drive_images / filename
    destination = local_images / filename
    assert source.is_file(), f'Selected raw source missing from Drive: {source}'
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists() or destination.stat().st_size != source.stat().st_size:
        temporary = destination.with_suffix(destination.suffix + '.part')
        shutil.copy2(source, temporary)
        temporary.replace(destination)
    return destination

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(stage_selected_source, row) for row in fixed_rows]
    for completed, future in enumerate(as_completed(futures), start=1):
        future.result()
        if completed % 250 == 0 or completed == len(futures):
            print(f'Staged {completed}/{len(futures)} selected raw sources')

with source_manifest.open(newline='') as stream:
    reader = csv.DictReader(stream)
    source_fields = reader.fieldnames
    selected_source_rows = [row for row in reader if row['source_id'] in selected_source_ids]
assert source_fields and 'source_path' in source_fields
assert len(selected_source_rows) == 2000, (
    f'Expected 2,000 selected source records, found {len(selected_source_rows)}'
)
for row in selected_source_rows:
    row['source_path'] = str(local_images / Path(row['source_path']).name)
local_source_manifest = PROJECT / 'artifacts/task2/source_manifest_split_local.csv'
local_source_manifest.parent.mkdir(parents=True, exist_ok=True)
with local_source_manifest.open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=source_fields)
    writer.writeheader()
    writer.writerows(selected_source_rows)

run_command(
    'python', 'scripts/build_matched_clean.py',
    '--source-manifest', local_source_manifest,
    '--output-root', PROJECT / 'artifacts/task2/matched_candidates',
    '--output-manifest', regenerated_manifest,
    '--report', PROJECT / 'artifacts/task2/fixed_q96_report_regenerated.json',
    '--policy', 'fixed_q96', '--seed', '42', '--limit-per-label', '1000',
)
with regenerated_manifest.open(newline='') as stream:
    regenerated_rows = list(csv.DictReader(stream))
fixed_by_source = {row['source_id']: row for row in fixed_rows}
regenerated_by_source = {row['source_id']: row for row in regenerated_rows}
assert regenerated_by_source.keys() == fixed_by_source.keys(), 'Regenerated source set changed'
byte_mismatches = [
    source_id for source_id in fixed_by_source
    if regenerated_by_source[source_id]['sha256'] != fixed_by_source[source_id]['sha256']
]
assert not byte_mismatches, (
    'Regenerated fixed-Q96 bytes differ from Notebook 07 inputs; stop before comparing '
    f'predictions. First mismatches: {byte_mismatches[:5]}'
)

combined_manifest = ROBUSTNESS / 'manifests/combined_manifest.csv'
clean_report_path = ROBUSTNESS / 'manifests/clean_manifest_report.json'
reuse_bank = False
if combined_manifest.is_file() and clean_report_path.is_file():
    clean_report = json.loads(clean_report_path.read_text())
    if clean_report.get('input_manifest_sha256') == sha256_file(regenerated_manifest):
        with combined_manifest.open(newline='') as stream:
            existing_rows = list(csv.DictReader(stream))
        reuse_bank = bool(existing_rows) and all(Path(row['image_path']).is_file() for row in existing_rows)
if reuse_bank:
    print(f'Reusing complete local robustness bank: {len(existing_rows):,} rows')
else:
    run_command(
        'make', 'robustness-prepare',
        f'TASK2_SELECTED_MANIFEST={regenerated_manifest}',
    )

parent_filenames = ('best_50_50.pt', 'best_50_50_predictions.csv')
for seed in SEEDS:
    durable_seed = DURABLE / 'train-controlled-rine' / f'seed_{seed}'
    local_seed = ROBUSTNESS / 'train-controlled-rine' / f'seed_{seed}'
    durable_parents = [durable_seed / filename for filename in parent_filenames]
    local_parents = [local_seed / filename for filename in parent_filenames]
    if all(path.is_file() for path in local_parents):
        print(f'Reusing local controlled-RINE seed {seed}')
    elif all(path.is_file() for path in durable_parents):
        local_seed.mkdir(parents=True, exist_ok=True)
        for source in durable_parents:
            shutil.copy2(source, local_seed / source.name)
        print(f'Restored controlled-RINE seed {seed} from Drive')
    else:
        missing_names = [path.name for path in durable_parents if not path.is_file()]
        print(
            f'Notebook 07 parent seed {seed} is incomplete on Drive ({missing_names}); '
            'retraining this parent locally.'
        )
        run_command('make', 'robustness-rine-train', f'ROBUSTNESS_SEED={seed}')
    for filename in parent_filenames:
        assert (local_seed / filename).is_file(), (
            f'Controlled-RINE parent seed {seed} did not produce {filename}'
        )

assert combined_manifest.is_file(), f'Combined robustness manifest missing: {combined_manifest}'
with combined_manifest.open(newline='') as stream:
    combined_rows = list(csv.DictReader(stream))
assert combined_rows, 'Combined robustness manifest is empty'
assert all(row['split'] != 'final_test' for row in combined_rows), 'final_test leaked into Notebook 08'
allowed_roots = (PROJECT / 'artifacts/task2/matched_candidates', ROBUSTNESS / 'variants')
unexpected = [
    row['image_path'] for row in combined_rows
    if not any(Path(row['image_path']).is_relative_to(root) for root in allowed_roots)
]
missing = [row['image_path'] for row in combined_rows if not Path(row['image_path']).is_file()]
assert not unexpected, f'Unexpected manifest paths: {unexpected[:5]}'
assert not missing, f'Missing manifest images: {missing[:5]} (total={len(missing)})'
print(f'PASS: local bank has {len(combined_rows):,} rows and all parent artifacts are restored.')


Staged 250/2000 selected raw sources
Staged 500/2000 selected raw sources
Staged 750/2000 selected raw sources
Staged 1000/2000 selected raw sources
Staged 1250/2000 selected raw sources
Staged 1500/2000 selected raw sources
Staged 1750/2000 selected raw sources
Staged 2000/2000 selected raw sources
Reusing complete local robustness bank: 20,850 rows
Reusing local controlled-RINE seed 42
Reusing local controlled-RINE seed 43
Reusing local controlled-RINE seed 44
PASS: local bank has 20,850 rows and all parent artifacts are restored.


In [18]:
import subprocess
from pathlib import Path

PROJECT = Path('/content/cya-techjam26')
ROBUSTNESS = PROJECT / 'artifacts/robustness'
SEEDS = (42, 43, 44)

def run_command(*args):
    result = subprocess.run(
        [str(value) for value in args],
        cwd=PROJECT,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): {' '.join(map(str, args))}"
        )

combined_manifest = ROBUSTNESS / 'manifests/combined_manifest.csv'
assert combined_manifest.is_file(), 'The prepared robustness bank is missing.'

for seed in SEEDS:
    seed_root = ROBUSTNESS / 'train-controlled-rine' / f'seed_{seed}'
    checkpoint = seed_root / 'best_50_50.pt'
    predictions = seed_root / 'best_50_50_predictions.csv'

    if checkpoint.is_file() and predictions.is_file():
        print(f'Reusing controlled-RINE seed {seed}')
        continue

    print(f'Retraining missing controlled-RINE seed {seed} on the active GPU...')
    run_command(
        'make',
        'robustness-rine-train',
        f'ROBUSTNESS_SEED={seed}',
    )
    assert checkpoint.is_file(), checkpoint
    assert predictions.is_file(), predictions

print('PASS: all controlled-RINE parents are ready.')

Reusing controlled-RINE seed 42
Reusing controlled-RINE seed 43
Reusing controlled-RINE seed 44
PASS: all controlled-RINE parents are ready.


In [19]:
from pathlib import Path

root = Path('/content/cya-techjam26/artifacts/robustness/train-controlled-rine')

for seed in (42, 43, 44):
    seed_root = root / f'seed_{seed}'
    print(
        seed,
        'complete:', (seed_root / 'complete.json').is_file(),
        'checkpoint:', (seed_root / 'best_50_50.pt').is_file(),
        'predictions:', (seed_root / 'best_50_50_predictions.csv').is_file(),
    )

42 complete: True checkpoint: True predictions: True
43 complete: True checkpoint: True predictions: True
44 complete: True checkpoint: True predictions: True


In [20]:
import shutil
from pathlib import Path

local_parents = Path(
    '/content/cya-techjam26/artifacts/robustness/train-controlled-rine'
)
drive_parents = Path(
    '/content/drive/MyDrive/cya-techjam26/artifacts/robustness/train-controlled-rine'
)

assert local_parents.is_dir(), local_parents
shutil.copytree(local_parents, drive_parents, dirs_exist_ok=True)
print('Saved controlled-RINE parents to Drive before PRNU extraction.')

Saved controlled-RINE parents to Drive before PRNU extraction.


In [21]:
import json
import subprocess
from pathlib import Path

PROJECT = Path('/content/cya-techjam26')
ROBUSTNESS = PROJECT / 'artifacts/robustness'
SEEDS = (42, 43, 44)

def run_make(target, *assignments):
    command = ['make', target, *assignments]
    result = subprocess.run(command, cwd=PROJECT, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(command)}")
    return result

combined_manifest = ROBUSTNESS / 'manifests/combined_manifest.csv'
assert combined_manifest.is_file(), 'Preparation did not produce the combined manifest'
signal_path = PROJECT / 'artifacts/task8b_v2/audits/prnu_v2_signal_validation.json'
assert signal_path.is_file(), f'256 px PREMIER signal report missing: {signal_path}'
signal_report = json.loads(signal_path.read_text())
assert signal_report.get('crop_size') == 256, 'Binary extraction requires 256 px signal validation'
assert signal_report.get('signal_validated'), 'The 256 px device-signal gate did not pass'
assert not signal_report.get('binary_authenticity_labels_used', False)
assert not signal_report.get('selection_or_heldout_rows_read', False)
for seed in SEEDS:
    parent = ROBUSTNESS / 'train-controlled-rine' / f'seed_{seed}' / 'best_50_50.pt'
    assert parent.is_file(), f'Controlled-RINE parent missing: {parent}'

extract_command = ['make', 'robustness-prnu-v2-extract']
extract_result = subprocess.run(extract_command, cwd=PROJECT, text=True)
readiness_path = ROBUSTNESS / 'features/prnu_v2_runtime_extraction_report.json'
assert readiness_path.is_file(), (
    f'PRNU-v2 extraction failed without a readiness report: {readiness_path}'
)
readiness = json.loads(readiness_path.read_text())
assert not readiness['final_test_read']
assert readiness['crop_size'] == 256
assert readiness['readiness_scope'] == 'matched_clean_rows_only'
print('PRNU-v2 readiness groups:', json.dumps(readiness['groups'], indent=2))
print('All-view PRNU coverage:', json.dumps(readiness['all_view_coverage_groups'], indent=2))

decision_path = ROBUSTNESS / 'reports/prnu_v2/retention_decision.json'
decision_path.parent.mkdir(parents=True, exist_ok=True)
if not readiness['ready_for_binary_ablation']:
    decision = {
        'candidate': 'rine_prnu_v2',
        'decision': 'blocked_data_readiness',
        'reason': 'Matched-clean Task 2 rows did not pass the predeclared native 256px readiness gate.',
        'required_crop_size': readiness['crop_size'],
        'groups': readiness['groups'],
        'all_view_coverage_groups': readiness['all_view_coverage_groups'],
        'prnu_only_training_run': False,
        'fusion_training_run': False,
        'retained_features': [],
        'reference_comparison_used': False,
        'final_test_read': False,
    }
    decision_path.write_text(json.dumps(decision, indent=2, sort_keys=True) + '\n')
    print(
        'STOP: PRNU-v2 is ineligible on this handoff. No PRNU diagnostic or fusion '
        'training was run. The readiness decision will be synced to Drive.'
    )
else:
    if extract_result.returncode != 0:
        raise RuntimeError(
            f'PRNU-v2 extraction failed after passing readiness: {extract_result.returncode}'
        )
    assert not readiness['reference_comparison_used']
    for seed in SEEDS:
        print(f'PRNU-only diagnostic seed {seed}')
        run_make('robustness-prnu-v2-train', f'ROBUSTNESS_SEED={seed}')
    for seed in SEEDS:
        print(f'RINE+PRNU fusion seed {seed}')
        run_make('robustness-prnu-v2-fusion', f'ROBUSTNESS_SEED={seed}')
    run_make('robustness-prnu-v2-compare')
    decision = json.loads(decision_path.read_text())
    assert decision['decision'] in {'retain', 'reject'}
assert not decision['final_test_read']
print(json.dumps(decision, indent=2, sort_keys=True))


PRNU-v2 readiness groups: [
  {
    "eligibility_rate": 1.0,
    "eligible_count": 607,
    "label": "authentic",
    "row_count": 607,
    "split": "seed_train"
  },
  {
    "eligibility_rate": 1.0,
    "eligible_count": 618,
    "label": "ai_generated",
    "row_count": 618,
    "split": "seed_train"
  },
  {
    "eligibility_rate": 1.0,
    "eligible_count": 76,
    "label": "authentic",
    "row_count": 76,
    "split": "selection_val"
  },
  {
    "eligibility_rate": 1.0,
    "eligible_count": 89,
    "label": "ai_generated",
    "row_count": 89,
    "split": "selection_val"
  }
]
All-view PRNU coverage: [
  {
    "eligibility_rate": 1.0,
    "eligible_count": 9105,
    "label": "authentic",
    "row_count": 9105,
    "split": "seed_train"
  },
  {
    "eligibility_rate": 1.0,
    "eligible_count": 9270,
    "label": "ai_generated",
    "row_count": 9270,
    "split": "seed_train"
  },
  {
    "eligibility_rate": 1.0,
    "eligible_count": 1140,
    "label": "authentic",
    "row_

In [22]:
import hashlib
import json
import shutil
from pathlib import Path

PROJECT = Path('/content/cya-techjam26')
ROBUSTNESS = PROJECT / 'artifacts/robustness'
DURABLE = Path('/content/drive/MyDrive/cya-techjam26/artifacts/robustness')
decision_path = ROBUSTNESS / 'reports/prnu_v2/retention_decision.json'
assert decision_path.is_file(), 'PRNU-v2 decision is missing; the ablation did not finish'
decision = json.loads(decision_path.read_text())

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def copy_file_verified(local_path, remote_path):
    remote_path.parent.mkdir(parents=True, exist_ok=True)
    local_hash = sha256_file(local_path)
    if not remote_path.is_file() or sha256_file(remote_path) != local_hash:
        temporary = remote_path.with_suffix(remote_path.suffix + '.part')
        shutil.copy2(local_path, temporary)
        temporary.replace(remote_path)
        copied = 1
    else:
        copied = 0
    assert sha256_file(remote_path) == local_hash, f'Drive verification failed: {remote_path}'
    return copied

output_paths = [
    Path('train-controlled-rine'),
    Path('features/prnu_v2_runtime_extraction_report.json'),
    Path('reports/prnu_v2'),
]
if decision['decision'] != 'blocked_data_readiness':
    output_paths.extend((
        Path('features/prnu_v2_runtime_features.csv'),
        Path('prnu_v2_runtime'),
        Path('rine_prnu_v2'),
    ))
copied = 0
verified = 0
for relative in output_paths:
    local_path = ROBUSTNESS / relative
    assert local_path.exists(), f'Expected Notebook 08 output missing: {local_path}'
    files = [local_path] if local_path.is_file() else sorted(
        path for path in local_path.rglob('*') if path.is_file()
    )
    for file_path in files:
        remote_path = DURABLE / file_path.relative_to(ROBUSTNESS)
        copied += copy_file_verified(file_path, remote_path)
        verified += 1
print(f'PASS: verified {verified} output files in {DURABLE}; copied {copied} new/changed files.')
print('The 19,460-image transform cache remains Colab-local by design.')


PASS: verified 67 output files in /content/drive/MyDrive/cya-techjam26/artifacts/robustness; copied 43 new/changed files.
The 19,460-image transform cache remains Colab-local by design.
